# Ligand-Pocket QGNN Training

This notebook trains the refactored **Ligand-Pocket QGNN** on the binding classification task.

**Architecture:**
- **Ligand**: Graph Neural Network (GCN) $\rightarrow$ Latent Vector
- **Pocket**: MLP $\rightarrow$ Latent Vector
- **Interaction**: Quantum Circuit (VQC) $\rightarrow$ Probability

**Task:** Binary Classification (Binding vs Non-Binding)

In [1]:
%load_ext autoreload
%autoreload 2

import os
import torch
import torch.nn as nn
import numpy as np
import matplotlib.pyplot as plt
from torch.utils.data import DataLoader
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score, accuracy_score
from tqdm.notebook import tqdm

from ligand_pocket_qgnn.data import LigandPocketDataProcessor, LigandPocketDataset, collate_fn
from ligand_pocket_qgnn.model import LigandPocketQGNN

# Configuration
DATA_DIR = "/media/priyanshu/SD/othercode/data"
SAVE_DIR = "./ligand_pocket_results"
os.makedirs(SAVE_DIR, exist_ok=True)

SEED = 42
np.random.seed(SEED)
torch.manual_seed(SEED)

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f"Using device: {DEVICE}")

Using device: cuda


## 1. Load Data

In [2]:
# Initialize Processor
processor = LigandPocketDataProcessor(DATA_DIR, seed=SEED)

# Load Data (Set max_samples=None for full dataset)
MAX_SAMPLES = 1000  # Adjust this for speed/full run
processor.load_data(max_samples=MAX_SAMPLES)

interactions = processor.get_dataset()
print(f"Total Interactions: {len(interactions)}")

Searching for data in: /media/priyanshu/SD/othercode/data
Found 1000 protein descriptor files


Loading Data: 100%|██████████| 1000/1000 [00:08<00:00, 124.13it/s]


Generating negative samples (target: 7264)...


Generating Negatives: 100%|██████████| 7264/7264 [00:04<00:00, 1619.63it/s]

Loaded 855 pockets, 7264 ligands
Interactions: 7264 positive, 7264 negative
Total Interactions: 14528


In [3]:
# Create Datasets and Loaders
train_ints, val_ints = train_test_split(interactions, test_size=0.2, random_state=SEED)

train_dataset = LigandPocketDataset(processor, train_ints)
val_dataset = LigandPocketDataset(processor, val_ints)

BATCH_SIZE = 32

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, collate_fn=collate_fn)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False, collate_fn=collate_fn)

print(f"Train batches: {len(train_loader)}")
print(f"Val batches: {len(val_loader)}")

Train batches: 364
Val batches: 91


## 2. Initialize Model

In [4]:
# Determine input dimensions from a sample
sample_ligand = processor.ligands[interactions[0].ligand_id]
sample_pocket = processor.pockets[interactions[0].pocket_id]

ligand_dim = sample_ligand.atom_features.shape[1]
pocket_dim = sample_pocket.to_vector().shape[0]

print(f"Ligand Input Dim: {ligand_dim}")
print(f"Pocket Input Dim: {pocket_dim}")

# Initialize Model
model = LigandPocketQGNN(
    ligand_in_dim=ligand_dim,
    pocket_in_dim=pocket_dim,
    hidden_dim=64,
    n_qubits=6,
    n_qlayers=2,
    use_quantum=True  # Set to False for Classical ablation
).to(DEVICE)

print(model)

Ligand Input Dim: 10
Pocket Input Dim: 19
LigandPocketQGNN(
  (ligand_encoder): LigandGNN(
    (conv1): GCNLayer(
      (linear): Linear(in_features=10, out_features=64, bias=True)
    )
    (conv2): GCNLayer(
      (linear): Linear(in_features=64, out_features=64, bias=True)
    )
    (lin): Linear(in_features=64, out_features=3, bias=True)
  )
  (pocket_encoder): PocketMLP(
    (net): Sequential(
      (0): Linear(in_features=19, out_features=64, bias=True)
      (1): ReLU()
      (2): Linear(in_features=64, out_features=64, bias=True)
      (3): ReLU()
      (4): Linear(in_features=64, out_features=3, bias=True)
    )
  )
  (interaction): QuantumInteractionLayer(
    (q_layer): <Quantum Torch Layer: func=circuit>
  )
)


## 3. Training Loop

In [5]:
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)
criterion = nn.BCELoss()

history = {
    'train_loss': [], 'train_acc': [],
    'val_loss': [], 'val_acc': [], 'val_auc': []
}

EPOCHS = 20
best_auc = 0.0

for epoch in range(EPOCHS):
    # --- TRAIN ---
    model.train()
    total_loss = 0
    all_preds = []
    all_labels = []
    
    pbar = tqdm(train_loader, desc=f"Epoch {epoch+1}/{EPOCHS}")
    for x_batch, edge_index_batch, batch_vec, pocket_batch, labels in pbar:
        x_batch = x_batch.to(DEVICE)
        edge_index_batch = edge_index_batch.to(DEVICE)
        batch_vec = batch_vec.to(DEVICE)
        pocket_batch = pocket_batch.to(DEVICE)
        labels = labels.to(DEVICE)
        
        optimizer.zero_grad()
        
        # Forward pass
        outputs = model(x_batch, edge_index_batch, batch_vec, pocket_batch).squeeze()
        
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        
        total_loss += loss.item()
        all_preds.extend(outputs.detach().cpu().numpy())
        all_labels.extend(labels.cpu().numpy())
        
        pbar.set_postfix({'loss': loss.item()})
        
    train_loss = total_loss / len(train_loader)
    train_acc = accuracy_score(all_labels, np.array(all_preds) >= 0.5)
    
    # --- VAL ---
    model.eval()
    val_preds = []
    val_labels = []
    val_loss_sum = 0
    
    with torch.no_grad():
        for x_batch, edge_index_batch, batch_vec, pocket_batch, labels in val_loader:
            x_batch = x_batch.to(DEVICE)
            edge_index_batch = edge_index_batch.to(DEVICE)
            batch_vec = batch_vec.to(DEVICE)
            pocket_batch = pocket_batch.to(DEVICE)
            labels = labels.to(DEVICE)
            
            outputs = model(x_batch, edge_index_batch, batch_vec, pocket_batch).squeeze()
            loss = criterion(outputs, labels)
            val_loss_sum += loss.item()
            
            val_preds.extend(outputs.cpu().numpy())
            val_labels.extend(labels.cpu().numpy())
            
    val_loss = val_loss_sum / len(val_loader)
    val_acc = accuracy_score(val_labels, np.array(val_preds) >= 0.5)
    try:
        val_auc = roc_auc_score(val_labels, val_preds)
    except:
        val_auc = 0.5
        
    # Update History
    history['train_loss'].append(train_loss)
    history['train_acc'].append(train_acc)
    history['val_loss'].append(val_loss)
    history['val_acc'].append(val_acc)
    history['val_auc'].append(val_auc)
    
    print(f"Epoch {epoch+1}: Train Loss={train_loss:.4f} Acc={train_acc:.4f} | Val Loss={val_loss:.4f} Acc={val_acc:.4f} AUC={val_auc:.4f}")
    
    if val_auc > best_auc:
        best_auc = val_auc
        torch.save(model.state_dict(), os.path.join(SAVE_DIR, "best_model_notebook.pt"))
        print("  Saved Best Model!")

ImportError: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html

## 4. Results Visualization

In [ ]:
plt.figure(figsize=(12, 4))

# Loss
plt.subplot(1, 2, 1)
plt.plot(history['train_loss'], label='Train Loss')
plt.plot(history['val_loss'], label='Val Loss')
plt.title('Loss')
plt.legend()

# Accuracy
plt.subplot(1, 2, 2)
plt.plot(history['train_acc'], label='Train Acc')
plt.plot(history['val_acc'], label='Val Acc')
plt.plot(history['val_auc'], label='Val AUC')
plt.title('Metrics')
plt.legend()

plt.show()